# TensorFlow 2.2 · tf.data 与时序窗口

按代码单元格的原编号查阅课程手册。

In [ ]:
import tensorflow as tf

# 用 from_tensor_slices 从对应的输入和标签构建数据集。
example = tf.data.Dataset.from_tensor_slices(([1., 2.], [2., 3.]))
print(next(iter(example)))

In [ ]:
import tensorflow as tf


class GetTrainTestData:
    def __init__(self, input_len, output_len, train_rate, is_train=True):
        series = tf.math.sin(tf.range(0., 1000., 0.1))
        split = int(len(series) * train_rate)
        # 先按时间切分，再在各自的片段内构造窗口。
        self.x = series[:split] if is_train else series[split:]
        self.input_len = input_len
        self.output_len = output_len
        self.sample_count = max(0, len(self.x) - input_len - output_len + 1)

    def __getitem__(self, index):
        if index < 0:
            index += self.sample_count
        if not 0 <= index < self.sample_count:
            raise IndexError(index)
        middle = index + self.input_len
        stop = middle + self.output_len
        return self.x[index:middle], self.x[middle:stop]

    def __len__(self):
        return self.sample_count

    def as_dataset(self):
        width = self.input_len + self.output_len
        windows = tf.signal.frame(self.x, width, frame_step=1)
        inputs = windows[:, :self.input_len]
        labels = windows[:, self.input_len:]
        return tf.data.Dataset.from_tensor_slices((inputs, labels))

In [ ]:
data_train = GetTrainTestData(input_len=3, output_len=1, train_rate=0.8, is_train=True)
data_test = GetTrainTestData(input_len=3, output_len=1, train_rate=0.8, is_train=False)

In [ ]:
src = tf.math.sin(tf.range(1., 1000., 0.1))
trg = tf.math.cos(tf.range(1., 1000., 0.1))

In [ ]:
data = tf.data.Dataset.from_tensor_slices((src, trg))
for idx, train in enumerate(data.batch(5).take(1)):
    print(idx, train)

In [ ]:
data_loader_train = data_train.as_dataset().batch(5)
data_loader_test = data_test.as_dataset().batch(5)

In [ ]:
xb, yb = next(iter(data_loader_train))
print('Windows:', len(data_train), len(data_test))
print('First batch:', tuple(xb.shape), tuple(yb.shape))